# Submission v10 — Lag features + targeted SMOTE + CatBoost+LGB blend

## Strategic rationale
v7c LOPO is stuck at 0.5130 because three nurses (P4DZ, 43JW, F1ZM, C8Q6) score at or below 0.50 chance level. P4DZ at 0.28 is *anti-predicting*. To push past 0.5130, we need a **structural change**, not just hyperparameter tuning (which would only give +0.005-0.015 and risks public LB overfit).

## Three combined fixes (all reversible if LOPO drops)

### 1. Within-session lag features (option B)
EDA showed stress almost never changes between consecutive 180s windows. Adding the previous label's sensor stats as lag features uses this directly. Restricted to within-session only (gap less than 10min between consecutive labels) — across-session lag would be noise.

### 2. Class 1-targeted SMOTE
v6 used global SMOTE and lost. The mistake was oversampling class 0 too. Class 0 already has 162 samples — enough. Class 1 has only 66 (8.1%) — that's what's killing P4DZ. Use `SMOTE(sampling_strategy={1: 200}, k_neighbors=3)` to only oversample class 1 to ~200 samples.

### 3. CatBoost + LightGBM blend
Two diverse learners on identical features. CatBoost handles small samples and ordered targets differently than LGB. Course covered both as gradient-boosted trees. Average their probability outputs before calibration.

## Gate
LOPO must be >= 0.5130 OR each component must be reversible. We'll print LOPO for the blend AND for each individual model, so we can pick the best of three submissions.

## Keep from v7c
- All v7c features (106 features: absolute stats + delta/slope/t1t3 + HRV time-domain + pid_enc + accel_mag)
- α=1.8 calibration
- 3 seeds x 5 folds ensemble
- class_weight='balanced' + capped sample weights


In [1]:
%pip install lightgbm catboost imbalanced-learn scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from catboost import CatBoostClassifier
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS = 180_000
HALF_MS   =  90_000
THIRD_MS  =  60_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']:
            f['hrv_'+k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn']/f['hrv_mean_rr'] if f['hrv_mean_rr']>1e-6 else 0.
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]
        for col in SENSOR_COLS:
            v   = wa[col].dropna().values.astype(float)
            vf  = wf[col].dropna().values.astype(float)
            vl  = wl[col].dropna().values.astype(float)
            vt1 = wt1[col].dropna().values.astype(float)
            vt3 = wt3[col].dropna().values.astype(float)
            if len(v)==0:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr',
                          'delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[col+'_'+s] = np.nan
                continue
            feat[col+'_mean']   = float(np.mean(v))
            feat[col+'_std']    = float(np.std(v))
            feat[col+'_min']    = float(np.min(v))
            feat[col+'_max']    = float(np.max(v))
            feat[col+'_median'] = float(np.median(v))
            feat[col+'_skew']   = float(spstats.skew(v)) if len(v)>2 else 0.
            feat[col+'_kurt']   = float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[col+'_range']  = float(np.max(v)-np.min(v))
            feat[col+'_q25']    = float(np.percentile(v,25))
            feat[col+'_q75']    = float(np.percentile(v,75))
            feat[col+'_iqr']    = float(np.percentile(v,75)-np.percentile(v,25))
            feat[col+'_delta']  = float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[col+'_slope']  = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.
            feat[col+'_t1_mean'] = float(np.mean(vt1)) if len(vt1)>0 else 0.
            feat[col+'_t3_mean'] = float(np.mean(vt3)) if len(vt3)>0 else 0.
            feat[col+'_t3t1']    = feat[col+'_t3_mean']-feat[col+'_t1_mean']

        # Accel magnitude
        if len(wa) > 0:
            ax = wa['accel_x'].fillna(0).values; ay = wa['accel_y'].fillna(0).values; az = wa['accel_z'].fillna(0).values
            mag = np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan

        # HRV time-domain
        feat.update(hrv_time_domain(wa['heart_rate']))

        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p:i for i,p in enumerate(TRAIN_LABEL['pid'].unique())}

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print(f'  train shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, train_pid_map)
print(f'  test shape: {test_features.shape}')


Extracting train features...
  train shape: (815, 106)
Extracting test features...
  test shape: (1028, 106)


## Within-session lag features (NEW in v10)

For each label, look up the **previous label's** sensor stats from the same nurse, but only if the previous label is within 600,000ms (10min) — otherwise it's a different session and lag would be noise. This adds 6 lag features per sensor (mean, std, delta), giving 36 new features.

In [4]:
LAG_LIMIT_MS = 600_000  # 10 minutes — beyond this gap, treat as different session
LAG_SENSORS = ['eda','heart_rate','temperature']  # the 3 most informative
LAG_STATS   = ['mean','std','delta']

def add_lag_features(features_df, label_df):
    """Add prev-window stats from same nurse if within session gap."""
    lab = label_df.copy().reset_index(drop=True)
    lab['timestamp'] = lab['timestamp'].astype(float)
    lab = lab.sort_values(['pid','timestamp']).reset_index(drop=True)
    
    # Build lag mapping: id -> id of previous in-session label
    prev_id_for = {}
    for _, grp in lab.groupby('pid'):
        ids = grp['id'].values
        ts  = grp['timestamp'].values
        for i in range(1, len(ids)):
            if (ts[i] - ts[i-1]) <= LAG_LIMIT_MS:
                prev_id_for[ids[i]] = ids[i-1]

    # Build new lag columns
    new_cols = {}
    for s in LAG_SENSORS:
        for st in LAG_STATS:
            new_cols[f'lag_{s}_{st}'] = []
            new_cols[f'lag_avail'] = new_cols.get('lag_avail', [])

    out = features_df.copy()
    avail_flag = []
    for lid in out.index:
        prev = prev_id_for.get(lid, None)
        if prev is not None and prev in features_df.index:
            avail_flag.append(1)
            for s in LAG_SENSORS:
                for st in LAG_STATS:
                    col = f'{s}_{st}'
                    out.loc[lid, f'lag_{col}'] = features_df.loc[prev, col]
        else:
            avail_flag.append(0)
            for s in LAG_SENSORS:
                for st in LAG_STATS:
                    out.loc[lid, f'lag_{s}_{st}'] = np.nan
    out['lag_avail'] = avail_flag
    return out

print('Adding lag features (within-session, ≤10min gap)...')
train_features = add_lag_features(train_features, TRAIN_LABEL)
test_features  = add_lag_features(test_features,  TEST_LABEL)

print(f'  train shape (with lag): {train_features.shape}')
print(f'  test shape  (with lag): {test_features.shape}')
print(f'  lag_avail rate train: {train_features["lag_avail"].mean()*100:.1f}%')
print(f'  lag_avail rate test:  {test_features["lag_avail"].mean()*100:.1f}%')


Adding lag features (within-session, ≤10min gap)...
  train shape (with lag): (815, 116)
  test shape  (with lag): (1028, 116)
  lag_avail rate train: 90.7%
  lag_avail rate test:  87.8%


In [5]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {0: total/(n_cls*counts[0]),
                 1: min(total/(n_cls*counts[1]), 2.5),
                 2: total/(n_cls*counts[2])}
sample_weights = np.array([class_weights[yi] for yi in y])
train_prior    = np.array([counts[i]/total for i in range(3)])

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class weights (capped):', {k: round(v,3) for k,v in class_weights.items()})


X_imp shape     : (815, 116)
X_test_imp shape: (1028, 116)
Class weights (capped): {0: 1.677, 1: 2.5, 2: 0.463}


## Class 1-targeted SMOTE wrapper (NEW in v10)

Only oversample class 1 (66 samples) up to 200, leave class 0 and class 2 alone. v6 made the mistake of global SMOTE — that polluted class 0 (which already has 162 samples). Targeted SMOTE on the genuine minority is what the imbalanced classification lecture actually recommends.

In [6]:
def safe_smote_class1(X, y, target_n=200, k=3):
    """Apply SMOTE only on class 1 to reach target_n samples. Returns (X_aug, y_aug)."""
    counts_local = Counter(y)
    if counts_local[1] >= target_n:
        return X, np.array(y)
    try:
        sm = SMOTE(sampling_strategy={1: target_n}, k_neighbors=k, random_state=42)
        Xs, ys = sm.fit_resample(X, y)
        return Xs, ys
    except Exception as e:
        print(f'  SMOTE failed: {e}, returning original')
        return X, np.array(y)


## LOPO CV — measure each model + blend separately

Print per-person LOPO for:
1. LightGBM only (baseline = v7c with lag features)
2. CatBoost only
3. Blend (avg probabilities)

Pick the best one to submit.

In [7]:
def lgb_train(X_tr, y_tr, sw_tr, seed=42):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
        class_weight='balanced', objective='multiclass', num_class=3,
        n_jobs=-1, verbose=-1, random_state=seed)
    m.fit(X_tr, y_tr, sample_weight=sw_tr, callbacks=[lgb.log_evaluation(-1)])
    return m

def cat_train(X_tr, y_tr, sw_tr, seed=42):
    cw = [class_weights[0], class_weights[1], class_weights[2]]
    m = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
        loss_function='MultiClass', class_weights=cw,
        random_seed=seed, verbose=False, allow_writing_files=False)
    m.fit(X_tr, y_tr, sample_weight=sw_tr)
    return m

logo = LeaveOneGroupOut()
lopo_lgb, lopo_cat, lopo_blend = [], [], []
print('=== LOPO CV — v10 (lag features + SMOTE class 1 + LGB+CAT blend) ===')
for tr_idx, va_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[va_idx[0]]
    y_tr_raw = y.iloc[tr_idx].values
    y_va     = y.iloc[va_idx].values
    if len(set(y_va)) < 2:
        print(f'  Skip {pid_val}: only one class')
        continue
    X_tr = X_imp.iloc[tr_idx].values
    X_va = X_imp.iloc[va_idx].values
    sw_tr_raw = sample_weights[tr_idx]

    # Apply class-1 SMOTE only on training fold
    X_tr_sm, y_tr_sm = safe_smote_class1(X_tr, y_tr_raw, target_n=200, k=3)
    sw_tr_sm = np.array([class_weights[yi] for yi in y_tr_sm])

    m_lgb = lgb_train(X_tr_sm, y_tr_sm, sw_tr_sm, seed=42)
    m_cat = cat_train(X_tr_sm, y_tr_sm, sw_tr_sm, seed=42)
    p_lgb = m_lgb.predict_proba(X_va)
    p_cat = m_cat.predict_proba(X_va)
    p_blend = 0.5*p_lgb + 0.5*p_cat

    s_lgb   = balanced_accuracy_score(y_va, np.argmax(p_lgb, axis=1))
    s_cat   = balanced_accuracy_score(y_va, np.argmax(p_cat, axis=1))
    s_blend = balanced_accuracy_score(y_va, np.argmax(p_blend, axis=1))
    print(f'  Leave out {pid_val}: LGB={s_lgb:.4f}  CAT={s_cat:.4f}  BLEND={s_blend:.4f}  (n={len(va_idx)})')
    lopo_lgb.append(s_lgb)
    lopo_cat.append(s_cat)
    lopo_blend.append(s_blend)

print()
print(f'v10 LGB-only LOPO   = {np.mean(lopo_lgb):.4f} +/- {np.std(lopo_lgb):.4f}')
print(f'v10 CAT-only LOPO   = {np.mean(lopo_cat):.4f} +/- {np.std(lopo_cat):.4f}')
print(f'v10 BLEND LOPO      = {np.mean(lopo_blend):.4f} +/- {np.std(lopo_blend):.4f}')
print(f'v7c reference LOPO  = 0.5130')

best_idx = int(np.argmax([np.mean(lopo_lgb), np.mean(lopo_cat), np.mean(lopo_blend)]))
best_name = ['LGB','CAT','BLEND'][best_idx]
print(f'\nBest model on LOPO: {best_name}')


=== LOPO CV — v10 (lag features + SMOTE class 1 + LGB+CAT blend) ===
  Leave out 43JW: LGB=0.5604  CAT=0.5000  BLEND=0.5440  (n=93)
  Leave out C8Q6: LGB=0.5000  CAT=0.4331  BLEND=0.4965  (n=152)
  Leave out DT5C: LGB=0.3571  CAT=0.5771  BLEND=0.3571  (n=90)
  Leave out F1ZM: LGB=0.4925  CAT=0.4776  BLEND=0.4925  (n=137)
  Leave out HDS9: LGB=0.2350  CAT=0.0000  BLEND=0.1923  (n=135)
  Leave out P4DZ: LGB=0.3095  CAT=0.2925  BLEND=0.3016  (n=144)
  Leave out TPQI: LGB=0.5786  CAT=0.4640  BLEND=0.5321  (n=64)

v10 LGB-only LOPO   = 0.4333 +/- 0.1229
v10 CAT-only LOPO   = 0.3920 +/- 0.1788
v10 BLEND LOPO      = 0.4166 +/- 0.1247
v7c reference LOPO  = 0.5130

Best model on LOPO: LGB


## Final ensemble — train and save THREE submissions
Save submissions for LGB, CAT, and BLEND. Pick the one with the best LOPO.

In [8]:
SEEDS = [42, 7, 123]
n_test = len(X_test_imp)

all_lgb_proba = []
all_cat_proba = []

print('=== Final ensemble: 3 seeds x 5 folds ===')
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_lgb_test, fold_cat_test = [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
        X_tr = X_imp.iloc[tr_idx].values
        y_tr = y.iloc[tr_idx].values
        X_va = X_imp.iloc[va_idx].values
        y_va = y.iloc[va_idx].values

        X_tr_sm, y_tr_sm = safe_smote_class1(X_tr, y_tr, target_n=200, k=3)
        sw_tr_sm = np.array([class_weights[yi] for yi in y_tr_sm])

        m_lgb = lgb_train(X_tr_sm, y_tr_sm, sw_tr_sm, seed=seed)
        m_cat = cat_train(X_tr_sm, y_tr_sm, sw_tr_sm, seed=seed)

        p_va_lgb = m_lgb.predict_proba(X_va)
        p_va_cat = m_cat.predict_proba(X_va)
        s_lgb = balanced_accuracy_score(y_va, np.argmax(p_va_lgb, axis=1))
        s_cat = balanced_accuracy_score(y_va, np.argmax(p_va_cat, axis=1))
        print(f'  Seed {seed} Fold {fold}: LGB={s_lgb:.4f}  CAT={s_cat:.4f}')

        fold_lgb_test.append(m_lgb.predict_proba(X_test_imp.values))
        fold_cat_test.append(m_cat.predict_proba(X_test_imp.values))

    all_lgb_proba.append(np.mean(fold_lgb_test, axis=0))
    all_cat_proba.append(np.mean(fold_cat_test, axis=0))

raw_lgb_proba = np.mean(all_lgb_proba, axis=0)
raw_cat_proba = np.mean(all_cat_proba, axis=0)
raw_blend     = 0.5*raw_lgb_proba + 0.5*raw_cat_proba
print()
print('Raw distributions:')
for name, p in [('LGB',raw_lgb_proba),('CAT',raw_cat_proba),('BLEND',raw_blend)]:
    pred = np.argmax(p, axis=1)
    print(f'  {name}: ', dict(Counter(pred)))


=== Final ensemble: 3 seeds x 5 folds ===
  Seed 42 Fold 1: LGB=0.7341  CAT=0.7569
  Seed 42 Fold 2: LGB=0.7529  CAT=0.7296
  Seed 42 Fold 3: LGB=0.7558  CAT=0.8156
  Seed 42 Fold 4: LGB=0.8152  CAT=0.7732
  Seed 42 Fold 5: LGB=0.8408  CAT=0.7835
  Seed 7 Fold 1: LGB=0.7712  CAT=0.7987
  Seed 7 Fold 2: LGB=0.7319  CAT=0.7944
  Seed 7 Fold 3: LGB=0.7809  CAT=0.7685
  Seed 7 Fold 4: LGB=0.8132  CAT=0.7758
  Seed 7 Fold 5: LGB=0.7194  CAT=0.6982
  Seed 123 Fold 1: LGB=0.8548  CAT=0.7723
  Seed 123 Fold 2: LGB=0.7110  CAT=0.6571
  Seed 123 Fold 3: LGB=0.8436  CAT=0.7853
  Seed 123 Fold 4: LGB=0.7972  CAT=0.8043
  Seed 123 Fold 5: LGB=0.7287  CAT=0.7538

Raw distributions:
  LGB:  {np.int64(2): 318, np.int64(1): 417, np.int64(0): 293}
  CAT:  {np.int64(0): 425, np.int64(1): 592, np.int64(2): 11}
  BLEND:  {np.int64(2): 207, np.int64(0): 325, np.int64(1): 496}


In [9]:
CALIB_ALPHA = 1.8

def calibrate_and_save(raw_proba, name):
    cal = raw_proba * (train_prior ** CALIB_ALPHA)
    cal = cal / cal.sum(axis=1, keepdims=True)
    preds = np.argmax(cal, axis=1).astype(int)
    sub = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds})
    fname = f'submission_v10_{name}.csv'
    sub.to_csv(fname, index=False)
    dist = dict(Counter(preds))
    print(f'  {name}: {fname}  dist={dist}')
    return preds

print('=== Calibrated submissions (alpha=1.8) ===')
preds_lgb   = calibrate_and_save(raw_lgb_proba, 'lgb')
preds_cat   = calibrate_and_save(raw_cat_proba, 'cat')
preds_blend = calibrate_and_save(raw_blend,     'blend')

print(f'\nTrain prior: 0={counts[0]/total*100:.1f}%  1={counts[1]/total*100:.1f}%  2={counts[2]/total*100:.1f}%')
print(f'\nSubmit the version whose LOPO score was highest:')
print(f'  LGB   LOPO = {np.mean(lopo_lgb):.4f}  ->  submission_v10_lgb.csv')
print(f'  CAT   LOPO = {np.mean(lopo_cat):.4f}  ->  submission_v10_cat.csv')
print(f'  BLEND LOPO = {np.mean(lopo_blend):.4f}  ->  submission_v10_blend.csv')
print(f'\nIf NONE beat v7c LOPO=0.5130, do not submit and tell Claude to try option A or D.')


=== Calibrated submissions (alpha=1.8) ===
  lgb: submission_v10_lgb.csv  dist={np.int64(2): 805, np.int64(0): 156, np.int64(1): 67}
  cat: submission_v10_cat.csv  dist={np.int64(2): 999, np.int64(0): 26, np.int64(1): 3}
  blend: submission_v10_blend.csv  dist={np.int64(2): 920, np.int64(0): 91, np.int64(1): 17}

Train prior: 0=19.9%  1=8.1%  2=72.0%

Submit the version whose LOPO score was highest:
  LGB   LOPO = 0.4333  ->  submission_v10_lgb.csv
  CAT   LOPO = 0.3920  ->  submission_v10_cat.csv
  BLEND LOPO = 0.4166  ->  submission_v10_blend.csv

If NONE beat v7c LOPO=0.5130, do not submit and tell Claude to try option A or D.
